# 05 · Mapas de profundidad con Video Depth Anything Small (S8)

**Historia:** SP-02.7 (spike de profundidad monocular). **Kernel:** `colibri`.

Procesa el video completo dos veces (original y volteado horizontalmente) y guarda los mapas en float16 en
`runs/profundidad/cache_vda/<corto>_{orig,flip}.npz` (función `mapas_vda` de `00_comun`, que también usa la herramienta), ya alineados con el frame original. El notebook 06 los lee con
el backend `vda` de `00_comun`.

- **Licencia (D-17):** solo el modelo Small (Apache-2.0). Base y Large son CC-BY-NC y no se usan.
- **Salida:** disparidad relativa, mayor = más cerca (al revés que YOLO26-Depth).
- **Resolución:** la de `config.yaml` → `vda.sizes`, de mayor a menor hasta que cabe en la GPU; la usada queda en `info.json`.
  S8 usó 392 (6 GB en la laptop, sin xFormers). **Cambiarla mueve `z_norm`:** en i5bE, hasta 0,46 entre 448 y 392.

En modo `artefactos` no recalcula los mapas que ya existan: los `.npz` pesan cientos de MB y no van en git.

In [ ]:
%run 00_comun.ipynb
VIDEO_LIST = ["i5bE"]          # cortos de datos_derivados/fuentes.csv; SP-02.9 añadirá otro video de cámara fija
CACHE = RUNS / "profundidad" / "cache_vda"; CACHE.mkdir(parents=True, exist_ok=True)
VIDEOS = {c: str(video_path(FUENTES.set_index("corto").loc[c].video_id)) for c in VIDEO_LIST}
SIZES = tuple(int(s) for s in os.environ.get("VDA_SIZES", ",".join(map(str, CFG["vda"]["sizes"]))).split(","))
print(VIDEOS, "| sizes", SIZES)


In [ ]:
pendientes = [f"{n}_{t}" for n in VIDEOS for t in ("orig", "flip") if not (CACHE / f"{n}_{t}.npz").exists()]
if pendientes or MODO == "recalcular":
    if MODO == "recalcular":
        print("modo recalcular: borra a mano los .npz que quieras rehacer; los existentes no se sobrescriben")
    mapas_vda(VIDEOS, CACHE, SIZES)
else:
    print("mapas ya calculados:", sorted(p.name for p in CACHE.glob("*.npz")))

In [ ]:
info = json.loads((CACHE / "info.json").read_text(encoding="utf-8")) if (CACHE / "info.json").exists() else {}
print(info)
for n in VIDEOS:
    d = np.load(CACHE / f"{n}_orig.npz")["maps"]
    n_frames = int(FUENTES.set_index("corto").loc[n].frames)
    print(n, d.shape, d.dtype, "| frames esperados", n_frames, "✓" if len(d) == n_frames else "✗")

## Conclusión (S8)

Los mapas de VDA Small tienen consistencia temporal (el modelo procesa el video completo), cosa que YOLO26-Depth
cuadro a cuadro no da (D-13). Se usan **solo** para leer la profundidad relativa en los 7 puntos (notebook 06).